In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, color
from skimage.feature import graycomatrix, graycoprops
from skimage.util import img_as_ubyte
from scipy.stats import skew, kurtosis

In [3]:
JSON_PATH = '../data/json/imd2020_index_edges.json'

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

sample_data = dict(list(data.items())[:3])
for img_folder, sample in sample_data.items():
    if 'manipulated' in sample:
        print(img_folder)
        print(sample)

1a07yi
{'original': 'IMD2020_edges\\1a07yi\\1a07yi_orig.png', 'manipulated': [{'image': 'IMD2020_edges\\1a07yi\\c8swtoq_0.png', 'mask': 'IMD2020_edges\\1a07yi\\c8swtoq_0_mask.png'}]}
1a16mu
{'original': 'IMD2020_edges\\1a16mu\\1a16mu_orig.png', 'manipulated': [{'image': 'IMD2020_edges\\1a16mu\\c8t64te_0.png', 'mask': 'IMD2020_edges\\1a16mu\\c8t64te_0_mask.png'}, {'image': 'IMD2020_edges\\1a16mu\\c8t9rsw_0.png', 'mask': 'IMD2020_edges\\1a16mu\\c8t9rsw_0_mask.png'}]}
1a1ogs
{'original': 'IMD2020_edges\\1a1ogs\\1a1ogs_orig.png', 'manipulated': [{'image': 'IMD2020_edges\\1a1ogs\\c8tf5mq_0.png', 'mask': 'IMD2020_edges\\1a1ogs\\c8tf5mq_0_mask.png'}]}


In [ ]:
list = [1, 2, 3]
list.append(4)
list.extend([5, 6])
list

In [ ]:
dict = {
    '1' : 'a',
    '2' : [
        {
            '3' : 'b',
            '4' : 'c'
        },
        {
            '5' : 'd',
            '6' : 'e'
        }
    ]
}

result = {}

for item in dict['2']:
    it = iter(item)
    first_value = item[next(it)]
    second_key = next(it)
    print(first_value, second_key)

In [ ]:
JSON_PATH = 'imd2020_index.json'
DATASET_ROOT = 'IMD2020'
CHECKPOINT = 'DexiNed/checkpoints/BIPED/10/10_model.pth'
OUTPUT_DIR = 'IMD2020_edges'

In [ ]:
with open(JSON_PATH, 'r') as f:
    data = json.load(f)

image_paths = []
mask_paths = []

for image_folder, sample in data.items():
    if 'original' in sample and 'manipulated' in sample:
        image_paths.append(sample['original'])

        for manip in sample['manipulated']:
            it = iter(manip)
            image_paths.append(manip[next(it)])
            mask_paths.append(manip[next(it)])

In [ ]:
mask_paths[-42:]

In [ ]:
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

array = matrix.ravel()
array

In [ ]:
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6]
])
matrix[:, 0].shape == 2

In [ ]:
MASK_PATH = 'IMD2020_edges/1a1ogs/c8tf5mq_0_mask.png'
mask = io.imread(MASK_PATH)
print(mask.shape)
if len(mask.shape) == 3:
        if mask.shape[2] == 4:
            mask = mask[:, :, :3]
        mask = color.rgb2gray(mask)
        image = img_as_ubyte(mask)
print(mask.shape)

In [ ]:
def mean(arr):
    
    n = len(arr)
    
    if n == 0:
        raise ValueError("The input array cannot be empty.")
    
    return sum(arr) / n

def std(arr, ddof=0):
    n = len(arr)
    m = mean(arr)

    if ddof == 1:
        n -= 1

    variance = sum((arr - m) ** 2) / n
    
    return variance ** 0.5
    

def skew_man(arr):
    n = len(arr)
    m = np.mean(arr)
    sd = np.std(arr, ddof=1)
    third_moment = np.mean((arr - m) ** 3)
    
    return third_moment / (sd ** 3) if sd > 1e-8 else 0

def kurtosis_man(arr, fisher=True, bias=True):
    n = len(arr)
    m = np.mean(arr)
    sd = np.std(arr, ddof=1)
    fourth_moment = np.mean((arr - m) ** 4)

    kurt = fourth_moment / (sd ** 4) if sd > 1e-8 else 0

    if not bias:
        e = arr - m
        m2 = np.mean(e ** 2)
        m4 = np.mean(e ** 4)

        k2 = (n / (n - 1)) * m2
        k4 = (n**2 * ((n + 1) * m4 - 3 * (n - 1) * m2**2)) / ((n - 1) * (n - 2) * (n - 3))

        kurt = k4 / (k2 ** 2) if k2 > 1e-8 else 0

    kurt -= 3 if fisher else kurt

    return kurt

def corrcoef(a) -> float:
    x = a[:-1]
    y = a[1:]

    mx = x.mean()
    my = y.mean()

    sx = x.std()
    sy = y.std()

    if sx < 1e-8 or sy < 1e-8:
        corr = 0.0
    else:
        corr = np.mean((x - mx) * (y - my)) / (sx * sy)

    return corr

In [ ]:
np.random.seed(27)
patch = np.random.rand(32, 32)
patch = patch.flatten()

In [ ]:

%timeit skew(patch)
%timeit skew_man(patch)
%timeit kurtosis(patch)
%timeit kurtosis_man(patch)

In [ ]:
%timeit np.mean(patch)
%timeit mean(patch)
%timeit np.std(patch, ddof=1)
%timeit std(patch, ddof=1)

In [ ]:
kurtosis(patch) - kurtosis_man(patch), skew(patch) - skew_man(patch)

In [ ]:
patch1 = patch[:-1]
patch2 = patch[1:]
%timeit np.corrcoef(patch1, patch2)[0,1]
%timeit corrcoef(patch)

In [ ]:
np.random.seed(27)
glcm = np.random.rand(2, 2, 1, 2)
glcm_sum = np.sum(glcm, axis=(0, 1), keepdims=True)
glcm, glcm_sum

In [ ]:
I, J = np.ogrid[:5, :5]
I - J

In [ ]:
I = np.arange(2).reshape((2, 1, 1, 1))
mean = np.sum(I * glcm, axis=(0, 1))
I * glcm, mean

In [ ]:
arr1 = np.array([4, 2, 1])
arr2 = np.array([-2, 3, 5])
mask = arr1 < 2 # [False, False, True]
mask[arr2 < 2] = True
mask

In [ ]:
%timeit graycoprops(glcm, prop='correlation')
%timeit graycoprops_man(glcm, prop='correlation')

In [4]:
def count_dict(data):
    count = 0
    if isinstance(data, dict):
        for key, value in data.items():
            if key == 'mask':
                continue
            count += count_dict(value)
    elif isinstance(data, list):
        for item in data:
            count += count_dict(item)
    else:
        count += 1
    return count


In [5]:
arr = np.array([])
for key, value in data.items():
    count = count_dict(value)
    arr = np.append(arr, count)
arr = arr.astype(np.int64)

In [66]:
print(
    arr[:45].sum(),
    arr[45:80].sum(),
    arr[80:110].sum(),
    arr[110:140].sum(),
    arr[140:170].sum(),
    arr[170:200].sum(),
    arr[200:240].sum(),
    arr[240:275].sum(),
    arr[275:310].sum(),
    arr[310:350].sum(),
    arr[350:].sum(),
)

216 222 216 229 224 215 244 223 235 205 195


In [6]:
feat_list = []
feat_list.append([1, 2, 3])
feat_list.append([4, 5, 6])
feat_edge = [7, 8, 9]
feat = np.hstack(feat_list)
feat = np.append(feat, feat_edge)
feat_final = np.hstack(feat)
feat_final

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [2]:
l1 = [1, 2, 3]
l2 = [4, 5, 6]
l = l1 + l2
l

[1, 2, 3, 4, 5, 6]